# Datamine ESTIMA Process Example

This notebook demonstrates univariate grade estimation (Nearest Neighbour, Inverse Distance, Ordinary Kriging) using the **`estima`** process wrapper in `dmstudio`.

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder)
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__')))

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Prepare Inputs
We copy the drillhole data and tutorial parameter files locally to this directory, and run `PROTOM` to define the prototype coordinates overlapping our samples.

In [ ]:
# Directories setup
notebook_folder = os.path.dirname(os.path.abspath('__file__'))
help_db = r"D:\Active\dmstudio\datamine_help\Database\DMTutorials\Data\VBOP\Datamine"

# Copy sample drillholes and parameter files locally
shutil.copy(r"D:\Active\dmstudio\tutorials\comp_holes.dmx", os.path.join(notebook_folder, "comp_holes.dmx"))
shutil.copy(os.path.join(help_db, "_vb_spar.dmx"), os.path.join(notebook_folder, "search_params.dmx"))
shutil.copy(os.path.join(help_db, "_vb_epar.dmx"), os.path.join(notebook_folder, "estimation_params.dmx"))
shutil.copy(os.path.join(help_db, "_vb_vpar.dmx"), os.path.join(notebook_folder, "kriging_vmodel.dmx"))

# Generate local model prototype
print("Generating block model prototype...")
dm_fil.protom(
    out_o=os.path.join(notebook_folder, 'model_proto'),
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)
print("Setup complete.")

## Step 3: Run ESTIMA Grade Estimation
We run the `estima` process to interpolate the AU, CU, and DENSITY grades into our block model cells using Ordinary Kriging.

In [ ]:
# Run ESTIMA
print("Running grade estimation (ESTIMA)...")
dm_cmd.estima(
    proto_i=os.path.join(notebook_folder, 'model_proto'),
    in_i=os.path.join(notebook_folder, 'comp_holes'),
    srcparm_i=os.path.join(notebook_folder, 'search_params'),
    estparm_i=os.path.join(notebook_folder, 'estimation_params'),
    vmodparm_i=os.path.join(notebook_folder, 'kriging_vmodel'),
    model_o=os.path.join(notebook_folder, 'grade_model_estima_result'),
    x_f='X',
    y_f='Y',
    z_f='Z'
)
print("ESTIMA completed successfully.")

## Step 4: Verify and Load Results in Pandas
Read the estimated model cells to print descriptive grade statistics.

In [ ]:
# Read outputs using read_datamine
result_file = os.path.join(notebook_folder, 'grade_model_estima_result.dmx')
shutil.copy(result_file, 'grade_model_estima_temp.dmx')
df_estima = agent.read_datamine('grade_model_estima_temp.dmx')
os.remove('grade_model_estima_temp.dmx')

print(f"ESTIMA block model cells: {len(df_estima)}")
print("
ESTIMA Gold (AU) Grade Summary:")
print(df_estima[df_estima['AU'] > -99]['AU'].describe())

## Step 5: Clean up Project Folder
Run this cell to clean up all copied and generated files, leaving only this notebook.

In [ ]:
# Cleanup local files
files_to_clean = [
    'comp_holes.dmx', 'search_params.dmx', 'estimation_params.dmx', 
    'kriging_vmodel.dmx', 'model_proto.dmx', 'grade_model_estima_result.dmx'
]
for f in files_to_clean:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed {f}")